# SPMamba - Test & Evaluation
Load `last.ckpt`, tách nguồn từ file mix, chấm điểm SDRi / SI-SNRi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba-root"
os.environ["PATH"] = f"{os.getcwd()}/bin:" + os.environ["PATH"]
!./bin/micromamba shell init -s bash -p /content/micromamba-root

In [ ]:
!./bin/micromamba create -p /content/spmamba-env python=3.10 --yes -q

!./bin/micromamba run -p /content/spmamba-env \
    pip install "setuptools<70" typeguard==2.13.3 h5py \
    pytorch-lightning==2.0.2 einops soundfile librosa \
    fast-bss-eval speechbrain==0.5.14 hydra-core rich \
    torch-complex packaging -q
!./bin/micromamba run -p /content/spmamba-env \
    pip install torch-mir-eval torch-optimizer -q
!./bin/micromamba run -p /content/spmamba-env pip install transformers -q

!./bin/micromamba run -p /content/spmamba-env \
    pip install torch==2.5.0 torchaudio==2.5.0 torchvision==0.20.0 \
    --index-url https://download.pytorch.org/whl/cu124 \
    --force-reinstall -q

!./bin/micromamba run -p /content/spmamba-env \
    pip install \
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.0/causal_conv1d-1.6.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" \
    "https://github.com/state-spaces/mamba/releases/download/v2.3.0/mamba_ssm-2.3.0+cu12torch2.5cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" \
    --no-deps --force-reinstall -q

!./bin/micromamba run -p /content/spmamba-env python -c "import torch; print('torch:', torch.__version__)"
!./bin/micromamba run -p /content/spmamba-env python -c "from mamba_ssm.modules.mamba_simple import Mamba; print('mamba-ssm: OK')"

In [ ]:
REPO_DIR = "/content/spmamba"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoanganh0106/spmamba.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

## Upload file mix + file nguồn gốc
Upload `mix.wav`, `s1.wav`, `s2.wav`, `s3.wav`

In [ ]:
from google.colab import files
import shutil

UPLOAD_DIR = "/content/test_audio"
# Xóa thư mục cũ để tránh file bị đổi tên (1)
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)
os.makedirs(UPLOAD_DIR)

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(os.path.join(UPLOAD_DIR, fname), 'wb') as f:
        f.write(data)
    print(f"  -> {UPLOAD_DIR}/{fname}")

## Tách nguồn + chấm điểm
Sửa đường dẫn `CKPT`, `MIX`, `SOURCES` nếu cần.

In [ ]:
CKPT = "/content/drive/MyDrive/SPMamba_checkpoints/SPMamba-3Speakers/last.ckpt"
MIX = "/content/test_audio/mix.wav"
SOURCES = "/content/test_audio/s1.wav /content/test_audio/s2.wav /content/test_audio/s3.wav"
OUTDIR = "/content/test_output"

!/content/bin/micromamba run -p /content/spmamba-env \
    python /content/spmamba/test_evaluate.py \
    --ckpt "{CKPT}" --mix "{MIX}" --sources {SOURCES} --outdir "{OUTDIR}"

## Nghe kết quả

In [ ]:
import IPython.display as ipd
import soundfile as sf

print("=== Mix ===")
w, sr = sf.read("/content/test_audio/mix.wav")
display(ipd.Audio(w, rate=sr))

for f in sorted(os.listdir("/content/test_output")):
    if f.endswith(".wav"):
        w, sr = sf.read(f"/content/test_output/{f}")
        print(f"\n=== {f} ===")
        display(ipd.Audio(w, rate=sr))

## Download kết quả

In [ ]:
from google.colab import files
for f in sorted(os.listdir("/content/test_output")):
    if f.endswith(".wav"):
        files.download(f"/content/test_output/{f}")